# Read a Feather file in Python with chDB (drop-in pandas)

Companion to [How to read a Feather file in Python (faster than pandas)](https://clickhouse.com/resources/engineering/read-feather-file-python).

chDB is a drop-in replacement for pandas: change one import line and your
existing pandas code runs on ClickHouse's engine, so it stays fast as files grow.

Run `./generate.sh` first to create `events.feather` (3M rows). Requirements: `pip install chdb pandas pyarrow`.

## 1. Read a Feather file into a DataFrame (types auto-inferred)

In [ ]:
import pandas as real_pd
import chdb.datastore as pd

real_pd.set_option("display.float_format", "{:.2f}".format)

df = pd.read_feather("events.feather")
df.head(8)

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
revenue = (df[df["event_type"] == "purchase"]
           .groupby("country")["amount"].sum()
           .sort_values(ascending=False))
revenue.to_pandas()

## 3. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 4. Performance: same code, one import swapped, on a 3M-row Feather file

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~1.5x faster than `pandas.read_feather`.

In [ ]:
import time

def datastore_agg():
    d = pd.read_feather("events.feather")
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["amount"].sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def pandas_agg():
    p = real_pd.read_feather("events.feather")
    return (p[p["event_type"] == "purchase"]
            .groupby("country")["amount"].sum()
            .sort_values(ascending=False))

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

pd_s = best_of_3(pandas_agg)
ds_s = best_of_3(datastore_agg)
print(f"import pandas as pd:              {pd_s:.3f}s")
print(f"import chdb.datastore as pd:      {ds_s:.3f}s")
print(f"speedup:                          {pd_s / ds_s:.1f}x")